In [2]:
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
df_student = pd.read_csv("StudentPerformanceFactors.csv")
df_student.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [4]:
df_student.describe()

,Hours_Studied,Attendance,Sleep_Hours,Previous_Scores,Tutoring_Sessions,Physical_Activity,Exam_Score
count,6607.000000,6607.000000,6607.00000,6607.000000,6607.000000,6607.000000,6607.000000
mean,19.975329,79.977448,7.02906,75.070531,1.493719,2.967610,67.235659
std,5.990594,11.547475,1.46812,14.399784,1.230570,1.031231,3.890456
min,1.000000,60.000000,4.00000,50.000000,0.000000,0.000000,55.000000
25%,16.000000,70.000000,6.00000,63.000000,1.000000,2.000000,65.000000
50%,20.000000,80.000000,7.00000,75.000000,1.000000,3.000000,67.000000
75%,24.000000,90.000000,8.00000,88.000000,2.000000,4.000000,69.000000
max,44.000000,100.000000,10.00000,100.000000,8.000000,6.000000,101.000000


In [5]:
# Select numeric columns to use for OLS (excluding the target 'Exam_Score')
numeric_cols = df_student.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Exam_Score') 

X = df_student[numeric_cols]
y = df_student['Exam_Score']

# Add a constant for the intercept
X = sm.add_constant(X)

# Fit the OLS model
ols_model = sm.OLS(y, X).fit()

# Show summary of OLS regression
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:             Exam_Score   R-squared:                       0.598
Model:                            OLS   Adj. R-squared:                  0.598
Method:                 Least Squares   F-statistic:                     1638.
Date:                Sat, 28 Mar 2026   Prob (F-statistic):               0.00
Time:                        10:50:19   Log-Likelihood:                -15338.
No. Observations:                6607   AIC:                         3.069e+04
Df Residuals:                    6600   BIC:                         3.074e+04
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                40.9271      0.33

In [8]:
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_student[numeric_cols])

# Split the data for evaluation
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

lasso = Lasso(alpha=0.1, random_state=42)
lasso.fit(X_train, y_train)

# Structured output
print("Lasso Regression Results")
print("------------------------")
print("Intercept: {:.4f}".format(lasso.intercept_))
print("Coefficients:")
for name, coef in zip(numeric_cols, lasso.coef_):
    print("  {:<20}: {:.4f}".format(name, coef))
print()
print("Train R^2 score: {:.4f}".format(lasso.score(X_train, y_train)))
print("Test R^2 score : {:.4f}".format(lasso.score(X_test, y_test)))

Lasso Regression Results
------------------------
Intercept: 67.2349
Coefficients:
  Hours_Studied       : 1.6316
  Attendance          : 2.1920
  Sleep_Hours         : -0.0000
  Previous_Scores     : 0.5929
  Tutoring_Sessions   : 0.5268
  Physical_Activity   : 0.0555

Train R^2 score: 0.5847
Test R^2 score : 0.6403
